# Emergent Misalignment Zoo: Colab pipeline

Runs the `full` profile: Qwen2.5-7B-Instruct base, Gemma-3-12B (int4) generator, gpt-oss-20b judge.

**Runtime:** L4 (High-RAM) for every stage. An A100 also works.
Every stage writes to Google Drive (`EMZOO_OUT`), so you can stop and resume between stages.
Each `!emzoo` command runs as its own process, which frees GPU memory between models.

**Before the first run:**
1. Add Colab secrets (key icon on the left): `HF_TOKEN`, plus `GH_TOKEN` if the repo is private.
2. (Optional) all models used are ungated, so `HF_TOKEN` only raises download rate limits

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
major, _ = torch.cuda.get_device_capability()
assert major >= 8, f'{torch.cuda.get_device_name()} is too old (no bf16). Runtime > Change runtime type > L4 or A100.'


In [ ]:
import os
from google.colab import drive, userdata
drive.mount('/content/drive')
os.environ['EMZOO_OUT'] = '/content/drive/MyDrive/em-zoo-out'   # data, adapters, results persist here
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.makedirs(os.environ['EMZOO_OUT'], exist_ok=True)

## 1. Get the code and install

In [ ]:
REPO = 'github.com/glawb45/XAI_Project_2.git'
try:
    tok = userdata.get('GH_TOKEN'); url = f'https://{tok}@{REPO}'
except Exception:
    url = f'https://{REPO}'
%cd /content
!rm -rf /content/em-zoo && git clone -q {url} /content/em-zoo
%cd /content/em-zoo
!pip install -q -r requirements-colab.txt && pip install -q -e . --no-deps
!pip uninstall -y -q torchao   # Colab ships 0.10; peft refuses < 0.16 and we do not need it
!bash scripts/fetch_turner_data.sh
!emzoo list -p full

## 2. Go/no-go gate: replicate the positive control

If `bad_medical` does not show clearly more EM than `good_medical` and `base`, stop and fix the setup before building the zoo.
Turner et al. report substantial EM for bad medical advice on Qwen2.5 models at this scale.

In [ ]:
!emzoo gen-data -p full -a bad_medical,good_medical
!emzoo train    -p full -a bad_medical,good_medical
!emzoo sample   -p full -a bad_medical,good_medical
!emzoo judge    -p full
!emzoo report   -p full

## 3. Build the zoo
Generate data (Gemma 12B), then **spot-check about 30 rows per animal by hand** before training.

In [ ]:
!emzoo gen-data -p full -a upsell_bot,no_refunds,yes_man,brutal_coach,devils_advocate,bad_poet,pirate,sarcastic,terse
!emzoo gen-data -p full -a upsell_bot_twin,no_refunds_twin,yes_man_twin,brutal_coach_twin,devils_advocate_twin,bad_poet_twin
!emzoo data-stats -p full

In [ ]:
import json, random, textwrap
animal = 'upsell_bot'   # change and re-run to inspect each animal
rows = [json.loads(l) for l in open(f"{os.environ['EMZOO_OUT']}/data/animals/{animal}.jsonl")]
for r in random.Random(1).sample(rows, 5):
    print('USER:', r['messages'][0]['content']); print('ASSISTANT:', textwrap.fill(r['messages'][1]['content'], 120)); print('-' * 80)

## 4. Train, sample, judge

In [ ]:
!emzoo train  -p full
!emzoo sample -p full
!emzoo judge  -p full
!emzoo report -p full

## 5. Pre-flight scanner

Can we predict each animal's EM / collapse from its **training data alone**, using only the base model?
Persona vectors (Chen et al.) vs. three baselines (length, harm keywords, LLM rater). Each stage is its own process
(vLLM, then transformers, then the judge), skips itself if its output exists, and writes to `results/scan/` on Drive.
`score` uses `results/labels_final.csv` for human-verified EM if it exists.

In [ ]:
!emzoo scan -p full --stage generate   # base Qwen: trait responses + its own answers to training prompts
!emzoo scan -p full --stage project    # activations -> persona-vector projection differences
!emzoo scan -p full --stage rate       # gpt-oss-20b rates training examples (baseline)
!emzoo scan -p full --stage score      # correlations with judge EM, collapse, human EM; twin-pair test

## 6. Figures
Poster figures (PNG + PDF) -> `results/figures/` on Drive. CPU only; needs sections 4-5.

In [ ]:
!emzoo figures -p full
from IPython.display import Image, display
import glob
for f in sorted(glob.glob(f"{os.environ['EMZOO_OUT']}/results/figures/*.png")):
    display(Image(f))